### Join fact to Dim

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
fact_order_cycle = spark.table("03_gold_catalog.dim_fact_tables.fact_order_cycle")
dim_store = spark.table("03_gold_catalog.dim_fact_tables.dim_store")
dim_technician = spark.table("03_gold_catalog.dim_fact_tables.dim_technician")
dim_estimator = spark.table("03_gold_catalog.dim_fact_tables.dim_estimator")
dim_vehicle = spark.table("03_gold_catalog.dim_fact_tables.dim_vehicle")
dim_date = spark.table("03_gold_catalog.dim_fact_tables.dim_date")
fact_invoice = spark.table("03_gold_catalog.dim_fact_tables.fact_invoice")
fact_survey = spark.table("03_gold_catalog.dim_fact_tables.fact_survey")

In [0]:
fact_order = fact_order_cycle.alias("f")
dim_store_df = dim_store.alias("ds")
dim_technician_df = dim_technician.alias("dt")
dim_estimator_df = dim_estimator.alias("de")
dim_vehicle_df = dim_vehicle.alias("dv")
dim_date_df = dim_date.alias("dd")

# Join using natural keys
fact_order_dim = (
    fact_order
    .join(dim_store_df, fact_order.store_id == dim_store_df.store_id, "left")
    .join(dim_technician_df, fact_order.technician_id == dim_technician_df.technician_id, "left")
    .join(dim_estimator_df, fact_order.estimator_id == dim_estimator_df.estimator_id, "left")
    .join(dim_vehicle_df, fact_order.vehicle_no == dim_vehicle_df.vehicle_no, "left")

    # Join date dimensions
    .join(dim_date_df.withColumnRenamed("date_key", "vehicle_in_date_key"),
          F.to_date(fact_order.vehicle_in_datetime) == dim_date_df.date, "left")

    .join(dim_date_df.alias("dd2").withColumnRenamed("date_key", "delivery_date_key"),
          fact_order.actual_delivery_datetime.cast("date") == F.col("dd2.date"), "left")
)

# Select final fields - natural keys + denormalized attributes
fact_order_dim = fact_order_dim.select(
    "f.order_id",
    "f.store_id",
    "f.technician_id",
    "f.estimator_id",
    "f.vehicle_no",
    
    "ds.store_name",
    "ds.manager_name",
    "dt.technician_name",
    "de.estimator_name",
    "dv.vehicle_make",
    "dv.vehicle_model",
    
    "vehicle_in_date_key",
    "delivery_date_key",
    
    "f.service_type",
    "f.order_status",
    "f.vehicle_in_datetime",
    "f.vehicle_out_datetime",
    "f.actual_work_start_datetime",
    "f.actual_completion_datetime",
    "f.actual_delivery_datetime",
    "f.vehicle_in_to_work_start_days",
    "f.work_start_to_completion_days",
    "f.work_completion_to_delivery_days",
    "f.days_in_shop",
    "f.planned_vs_actual_completion_days",
    "f.promised_vs_actual_delivery_days",
    
    "f.initial_estimate_amount",
    "f.final_estimate_amount"
)

print(f"fact_order_dim: {fact_order_dim.count()} rows")
display(fact_order_dim)

# Write back to Gold Layer
fact_order_dim.write.format("delta").mode("overwrite").saveAsTable("03_gold_catalog.datacube.fact_order_dim")

In [0]:
fact_inv = fact_invoice.alias("f")
dim_store_df = dim_store.alias("ds")
dim_technician_df = dim_technician.alias("dt")
dim_date_df = dim_date.alias("dd")

fact_invoice_dim = (
    fact_inv
    .join(dim_store_df, fact_inv.store_id == dim_store_df.store_id, "left")
    .join(dim_technician_df, fact_inv.technician_id == dim_technician_df.technician_id, "left")
    .join(dim_date_df,
          fact_inv.invoice_date == dim_date_df.date, "left")
)

fact_invoice_conformed = fact_invoice_dim.select(
    "f.invoice_id",
    "f.order_id",
    "f.store_id",
    "f.technician_id",
    
    "ds.store_name",
    "ds.manager_name",
    "dt.technician_name",
    
    F.col("dd.date_key").alias("invoice_date_key"),
    F.col("dd.year").alias("invoice_year"),
    F.col("dd.month").alias("invoice_month"),
    
    "f.invoice_amount",
    "f.payment_mode",
    "f.currency",
    "f.invoice_date",
    "f.invoice_ts"
)

print(f"fact_invoice_dim: {fact_invoice_conformed.count()} rows")
display(fact_invoice_conformed)

fact_invoice_conformed.write.format("delta").mode("overwrite").saveAsTable("03_gold_catalog.datacube.fact_invoice_dim")

In [0]:
fact_s = fact_survey.alias("f")
dim_store_df = dim_store.alias("ds")
dim_technician_df = dim_technician.alias("dt")
dim_date_df = dim_date.alias("dd")

fact_survey_dim = (
    fact_s
    .join(dim_store_df, fact_s.store_id == dim_store_df.store_id, "left")
    .join(dim_technician_df, fact_s.technician_id == dim_technician_df.technician_id, "left")

    .join(dim_date_df.withColumnRenamed("date_key", "survey_sent_date_key"),
          fact_s.survey_sent_date == dim_date_df.date, "left")

    .join(dim_date_df.alias("dd2").withColumnRenamed("date_key", "survey_response_date_key"),
          fact_s.survey_response_date == F.col("dd2.date"), "left")
)

fact_survey_conformed = fact_survey_dim.select(
    "f.survey_id",
    "f.order_id",
    "f.store_id",
    "f.technician_id",
    
    "ds.store_name",
    "dt.technician_name",
    
    "survey_sent_date_key",
    "survey_response_date_key",
    
    "f.responded_flag",
    "f.survey_sent_date",
    "f.survey_response_date",
    "f.overall_satisfaction_rating",
    "f.cleanliness_rating",
    "f.communication_rating",
    "f.work_quality_rating",
    "f.delivered_on_time_rating",
    "f.days_to_respond"
)
display(fact_survey_conformed)
print(f"fact_survey_dim: {fact_survey_conformed.count()} rows")

fact_survey_conformed.write.format("delta").mode("overwrite").saveAsTable("03_gold_catalog.datacube.fact_survey_dim")

In [0]:
# # Columns to drop to avoid duplicates
# invoice_cols_to_drop = ["order_id", "store_id", "technician_id", "store_name", "manager_name", "technician_name"]
# survey_cols_to_drop = ["order_id", "store_id", "technician_id", "store_name", "technician_name"]

# # Preview join results
# ssot_datacube = (
#     fact_order_dim
#     .join(fact_invoice_conformed.drop(*invoice_cols_to_drop), 
#           fact_order_dim.order_id == fact_invoice_conformed.order_id, 
#           "left")
#     .join(fact_survey_conformed.drop(*survey_cols_to_drop), 
#           fact_order_dim.order_id == fact_survey_conformed.order_id, 
#           "left")
# )

# display(ssot_datacube)

# ssot_datacube.write.format("delta").mode("overwrite").saveAsTable("03_gold_catalog.datacube.ssot")